In [ ]:
# Practice Problems Day 10

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 10 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice today's code before the weekly homework. Weekly Homework 5 assumes you have done them. Plan on about 45 minutes.

**Reading:** Geron chapter 10, **Implementing a Regression MLP** and **Implementing Mini-Batch Gradient Descent using DataLoaders**.

GOALS:

1. Build an MLP with `nn.Sequential` and count what it has to learn.
2. Write the mini-batch training loop yourself.
3. Explain batches, steps and epochs in words.

**How much of this should you write yourself?** Problem 1 gives you most of the code, with comments telling you what to write on the missing lines. Problem 2 lists the steps. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious.

**Save your work.** Commit and push when you are done, so you have your code for the weekly homework and the exam:

```bash
git add .
git commit -m "Day 10 practice"
git push
```

---

In [ ]:
# RUN THIS. The California housing data from the notes, 
# split, scaled, and turned into tensors.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

housing = pd.read_csv("data/california_housing.csv")
X = housing.drop(columns=["MedHouseVal"])
y = housing["MedHouseVal"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42)

preprocessing = Pipeline([("scale", StandardScaler())])      # the preprocessing pipeline, fit on the training rows only
X_train_scaled = preprocessing.fit_transform(X_train)
X_valid_scaled = preprocessing.transform(X_valid)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_valid_t = torch.tensor(X_valid_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_valid_t = torch.tensor(y_valid.values, dtype=torch.float32).reshape(-1, 1)

print("training rows:", X_train_t.shape[0], "   validation rows:", X_valid_t.shape[0])

## Problem 1: A deeper MLP

**What to copy, what to write:** Most of the code is given. Write the lines the comments ask for.

The reading uses an MLP with **two** hidden layers: 50 neurons, then 40 neurons, then the output.

**1a.** By hand, count the weights and biases in each of the three `nn.Linear` layers, and the total.

**1b.** Build the model in the cell below and check your count.

In [ ]:
torch.manual_seed(42)

# Build the model with nn.Sequential:
#   a Linear layer from 8 inputs to 50 neurons, then ReLU,
#   a Linear layer from 50 to 40, then ReLU,
#   a Linear layer from 40 to 1 output
deep_model = nn.Sequential(
    nn.Linear(8, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1))

print(deep_model)

# Count the numbers the model has to learn (this part is written for you)
n_params = 0
for p in deep_model.parameters():
    n_params = n_params + p.numel()
print("numbers to learn:", n_params)

**1a.** First layer $8 \times 50 + 50 = 450$. Second layer $50 \times 40 + 40 = 2040$. Output layer $40 + 1 = 41$. Total **2531**, about five times the 501 in the notes.

**1b.** The code prints 2531.

---

## Problem 2: Train it with mini-batches (steps outlined)

**What to copy, what to write:** The mini-batch loop is the same pattern as the notes. Make sure you understand it.

**2a.** Train `deep_model` with mini-batches of 32 rows, `torch.optim.SGD` with a learning rate of 0.01, and `nn.MSELoss`, for 20 epochs. After every epoch, save the training and validation RMSE. Plot both against the epoch.

**2b.** The one hidden layer MLP in the notes got a validation RMSE of 0.6067 after 20 epochs. Does the deeper model do better? Is it worth having five times as many numbers to learn?

In [ ]:
# Step 1 (2a): a TensorDataset and a DataLoader with batch_size=32 and shuffle=True
train_data = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

# Step 2 (2a): loss_fn = nn.MSELoss() and an SGD optimizer for deep_model.parameters() with lr=0.01
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(deep_model.parameters(), lr=0.01)

# Step 3 (2a): two empty lists for the RMSEs, then the loop:
#              for each of 20 epochs, for each batch: zero_grad, predict, loss, backward, step
#              after each epoch, inside torch.no_grad(), save the training and validation RMSE
#              (the RMSE is torch.sqrt of the MSE loss, then .item())
train_rmse = []
valid_rmse = []
for epoch in range(20):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = deep_model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        train_rmse.append(torch.sqrt(loss_fn(deep_model(X_train_t), y_train_t)).item())
        valid_rmse.append(torch.sqrt(loss_fn(deep_model(X_valid_t), y_valid_t)).item())

# Step 4 (2a): plot the two RMSE lists against the epoch, with labels and a legend
plt.figure(figsize=(6, 4))
plt.plot(range(1, 21), train_rmse, label="training")
plt.plot(range(1, 21), valid_rmse, label="validation")
plt.xlabel("epoch")
plt.ylabel("RMSE")
plt.legend()
plt.grid()
plt.show()

# Step 5 (2b): print the final validation RMSE
print("deep model validation RMSE after 20 epochs:", round(valid_rmse[-1], 4))
print("training RMSE after 20 epochs:            ", round(train_rmse[-1], 4))


**2a.** Both curves go down together, and after 20 epochs the training RMSE is 0.5588 and the validation RMSE is **0.568**.

**2b.** Yes, it does better: 0.568 against 0.6067, so the predictions are typically about \$3,900 closer. The training and validation RMSE are still close, so the extra numbers are not causing overfitting here. It still trains in seconds, so on this data it is worth it. So weigh what you gain against what it costs.

---

## Problem 3: In words

**3a.** Explain to a friend in DATA 101 what a **batch**, a **step** and an **epoch** are, using the numbers from today: 15,480 training rows and a batch size of 32. Three or four sentences, no code.

**3b.** A classmate's training loop prints `nan` for the loss after a few steps. Name some things they should check first, and say why each one could cause it. HINT - This could come from many parts of the code beginning to end.

**3a.** Your answer will be different, here is one way to say it: "A batch is a small group of rows, here 32 houses. A step uses one batch to compute the gradient and update every weight once. An epoch is one trip through all of the training data, so with 15,480 rows and batches of 32 that is 484 steps. Training for 20 epochs means every house gets used 20 times."

**3b.** (1) The **learning rate** is too big, so each step overshoots and the weights grow until the numbers overflow (the notes' 0.5 example). (2) The features are **not scaled**, so some inputs are in the thousands and the gradients are huge. Other good answers: forgetting `zero_grad` so the gradients pile up, or a missing value (NaN) in the data.

---

## Optional exploration

Only if you are curious. Nothing here is needed for the weekly homework.

**A.** Train the one hidden layer MLP from the notes with a batch size of **256** instead of 32, for 20 epochs. How many steps are in one epoch now? Is the final validation RMSE better or worse, and why?

**B.** Try a hidden layer of 200 neurons instead of 50. Does it help?

In [ ]:
results = []
for batch_size, hidden in [(256, 50), (32, 200)]:
    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(8, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
    for epoch in range(20):
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
    with torch.no_grad():
        rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()
    print("batch size", batch_size, "  hidden", hidden, "  steps per epoch", len(loader), "  validation RMSE", round(rmse, 4))

**A.** 15,480 / 256 is 61 steps per epoch (the last batch is smaller), so 20 epochs is only 1,220 steps instead of 9,680. With the same learning rate it makes far fewer updates, and it ends worse: validation RMSE 0.6777. Bigger batches give smoother steps but fewer of them, so they usually need more epochs or a bigger learning rate.

**B.** 200 neurons gives 0.5844, better than 50 neurons (0.6067) but not as good as the two layer model (0.568).